# Floor / Reach Calibration

Run this in the **same kernel** as `magpie_demo` (use *Select Kernel → existing kernel*).  
Requires `node`, `home`, and `GRIPPER_LEN` to already be defined from magpie_demo cells 1–5.

| Cell | Does |
|------|------|
| 1 | Verify node + home are available |
| 2 | Compute target floor TCP Z |
| 3 | Step-down to floor target — stops there, returns home |
| 4 | Diagram |
| 5 | Grasp reach table |

In [1]:
# ── Verify magpie_demo context is available ───────────────────────────────────
STEP_MM = 3   # mm per descent step

assert 'node' in dir(),        'node not found — run magpie_demo cells 1-4 first'
assert 'home' in dir(),        'home not found — run magpie_demo cell 5 first'
assert 'GRIPPER_LEN' in dir(), 'GRIPPER_LEN not found — run magpie_demo cell 3 first'
assert 'HARD_FLOOR_Z' in dir(),'HARD_FLOOR_Z not found — run magpie_demo cell 3 first'

node.spin(5)
v0 = poses.pose_mtrx_to_vec(home)
print(f'Home TCP z       : {v0[2]*1000:.1f} mm')
print(f'Home fingertip z : {(v0[2]-GRIPPER_LEN)*1000:.1f} mm')
print(f'HARD_FLOOR_Z     : {HARD_FLOOR_Z*1000:.1f} mm')
print('OK — ready to run step-down')

AssertionError: node not found — run magpie_demo cells 1-4 first

In [ ]:
# ── Record home + compute target ─────────────────────────────────────────────
fnode.spin(5)
home = fnode.tcp.copy()
v0   = poses.pose_mtrx_to_vec(home)

home_tcp_z  = v0[2]
home_fing_z = home_tcp_z - GRIPPER_LEN

# Lowest TCP z the arm would ever be commanded during a real grasp:
#   fingertip at HARD_FLOOR_Z  →  TCP = HARD_FLOOR_Z + GRIPPER_LEN
target_tcp_z  = HARD_FLOOR_Z + GRIPPER_LEN
target_fing_z = HARD_FLOOR_Z
total_descent = home_tcp_z - target_tcp_z
n_steps       = int(np.ceil(total_descent / (STEP_MM / 1000.)))

print(f'Home TCP z         : {home_tcp_z*1000:.1f} mm')
print(f'Home fingertip z   : {home_fing_z*1000:.1f} mm')
print(f'HARD_FLOOR_Z       : {HARD_FLOOR_Z*1000:.1f} mm')
print(f'Target TCP z       : {target_tcp_z*1000:.1f} mm  (fingertip exactly at floor)')
print(f'Total descent      : {total_descent*1000:.1f} mm')
print(f'Steps needed       : {n_steps}  ({STEP_MM} mm each)')

In [ ]:
# ── Step-down to floor target ─────────────────────────────────────────────────
step_m   = STEP_MM / 1000.
log      = []
cur_pose = home.copy()

node.unteach()

print(f'Descending to fingertip z = {HARD_FLOOR_Z*1000:.0f} mm  (TCP = {target_tcp_z*1000:.1f} mm)')
print(f'Step = {STEP_MM} mm\n')

while True:
    next_tcp_z = cur_pose[2, 3] - step_m
    if next_tcp_z < target_tcp_z:
        next_tcp_z = target_tcp_z

    cur_pose = cur_pose.copy()
    cur_pose[2, 3] = next_tcp_z
    tcp_z  = next_tcp_z
    fing_z = tcp_z - GRIPPER_LEN

    node.move(cur_pose, spd=0.03)
    node.spin(4)

    log.append({'tcp_z': tcp_z, 'fing_z': fing_z})
    actual_z = poses.pose_mtrx_to_vec(node.tcp)[2]
    marker = '<<< FLOOR TARGET' if abs(fing_z - HARD_FLOOR_Z) < 0.001 else ''
    print(f'  cmd={tcp_z*1000:6.1f}mm  actual={actual_z*1000:6.1f}mm  fingertip={fing_z*1000:6.1f}mm  {marker}')

    if tcp_z <= target_tcp_z:
        print('\nReached floor target.')
        break

    time.sleep(0.2)

print('\nReturning home...')
node.move(home, spd=0.05)
print('Done.')

In [ ]:
# ── Diagram — TCP z and fingertip z at every step ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

steps   = list(range(1, len(log) + 1))
tcp_zs  = [e['tcp_z']  * 1000 for e in log]
fing_zs = [e['fing_z'] * 1000 for e in log]

ax.plot(steps, tcp_zs,  'o-', color='royalblue', lw=2, label='TCP flange z')
ax.plot(steps, fing_zs, 's-', color='limegreen', lw=2, label='Fingertip z')

ax.axhline(HARD_FLOOR_Z * 1000, color='red', ls='--', lw=2,
           label=f'HARD_FLOOR_Z = {HARD_FLOOR_Z*1000:.0f} mm  ← floor target')
ax.axhline(target_tcp_z * 1000, color='royalblue', ls=':', lw=1.5, alpha=0.5,
           label=f'Target TCP = {target_tcp_z*1000:.1f} mm')

ax.set_xlabel('Step'); ax.set_ylabel('Z (mm)')
ax.set_title('Step-Down — TCP and Fingertip Z at Each Step')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Grasp reach table ─────────────────────────────────────────────────────────
# For objects of various heights sitting on the table, compute:
#   centroid_z  = TABLE_Z + height/2
#   auto_offset = clip(height/4, 5mm, 15mm)
#   gfz         = centroid_z - auto_offset   (fingertip at grasp)
#   PASS if gfz >= HARD_FLOOR_Z
#
# Set TABLE_Z to the measured table surface Z in the robot frame.
TABLE_Z = HARD_FLOOR_Z   # adjust if table surface != floor limit

heights_mm = [20, 30, 40, 50, 60, 70, 80, 100]

rows = []
for h_mm in heights_mm:
    h          = h_mm / 1000.
    centroid_z = TABLE_Z + h / 2.
    auto_off   = float(np.clip(h / 4., 0.005, 0.015))
    gfz        = centroid_z - auto_off
    ok         = gfz >= HARD_FLOOR_Z
    margin_mm  = (gfz - HARD_FLOOR_Z) * 1000.
    rows.append((h_mm, centroid_z*1000, auto_off*1000, gfz*1000, margin_mm, ok))

print(f'Table Z = {TABLE_Z*1000:.1f} mm    HARD_FLOOR_Z = {HARD_FLOOR_Z*1000:.1f} mm\n')
print(f'{"Height":>8}  {"Centroid Z":>10}  {"Offset":>7}  {"Grasp Z":>8}  {"Margin":>8}  Status')
print('-' * 62)
for h_mm, cz, off, gz, mg, ok in rows:
    print(f'{h_mm:>6}mm  {cz:>9.1f}mm  {off:>6.1f}mm  {gz:>7.1f}mm  {mg:>+7.1f}mm  {"PASS" if ok else "FAIL"}')

fig, ax = plt.subplots(figsize=(11, 5))
h_vals  = [r[0] for r in rows]
gz_vals = [r[3] for r in rows]
cols    = ['limegreen' if r[5] else 'tomato' for r in rows]

bars = ax.bar(h_vals, gz_vals, width=6, color=cols, alpha=0.85)
ax.axhline(HARD_FLOOR_Z * 1000, color='red',    ls='--', lw=2, label=f'HARD_FLOOR_Z {HARD_FLOOR_Z*1000:.0f}mm')
ax.axhline(TABLE_Z      * 1000, color='sienna', ls=':',  lw=2, label=f'Table surface {TABLE_Z*1000:.0f}mm')

for bar, (h_mm, cz, off, gz, mg, ok) in zip(bars, rows):
    ax.text(bar.get_x() + bar.get_width()/2, gz + 0.5,
            f'{gz:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Object height (mm)')
ax.set_ylabel('Fingertip Z at grasp (mm)')
ax.set_title('Grasp Reach by Object Height')
ax.legend(); plt.tight_layout(); plt.show()